In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import json
from scipy import stats
from nba_api.stats.endpoints import leaguedashteamstats
from datetime import datetime

pd.set_option('display.max_columns', None)

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.models.xgboost_model import *
from src.models.ngboost_model import *
from src.points_model import PointsPropModel
from src.utils.helper_functions import findOpp
from src.utils.team_info import projectedStartingFive, mainStartingFive, teamStarPlayer, nameDict, questionablePlayers

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'DAL': ['Brandon Williams', 'Daniel Gafford', 'Anthony Davis'], 'HOU': ['Tari Eason'], 'DEN': ['Julian Strawther'], 'LAC': ['Nicolas Batum', 'James Harden']}

Out Players:
{'BOS': ['Jayson Tatum'], 'TOR': ['RJ Barrett'], 'MIA': ['Pelle Larsson', 'Terry Rozier'], 'DAL': ["D'Angelo Russell", 'Kyrie Irving'], 'UTA': ['Jusuf Nurkić', 'Georges Niang'], 'HOU': ['Dorian Finney-Smith', 'Fred VanVleet'], 'DEN': ['Aaron Gordon', 'Christian Braun'], 'MEM': ['Ty Jerome', 'Scotty Pippen', 'Zach Edey', 'John Konchar', 'Javon Small'], 'LAC': ['Chris Paul', 'Derrick Jones']}
Note: DAL (Mavericks) has 3 confirmed players - lineup will still be updated
Note: LAC (Clippers) has 4 confirmed players - lineup will still be updated
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 10 teams with confirmed lineups
Updated 4 teams with questionable players


In [3]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

us_df = pd.read_csv(us_file)
us_points = us_df[us_df['CATEGORY'] == 'player_points'].copy()

dfs_df = pd.read_csv(dfs_file)
prizepicks_lines = dfs_df[dfs_df['BOOKMAKER'] == 'Underdog'].copy()
pp_points = prizepicks_lines[prizepicks_lines['CATEGORY'] == 'player_points'].copy()
print(f"Found {len(pp_points)//2} PrizePicks player point props")

print(f"DFS earliest pull: {dfs_df['DATA_PULLED_AT'].min()}")
print(f"DFS latest pull: {dfs_df['DATA_PULLED_AT'].max()}")
print(f"US earliest pull: {us_df['DATA_PULLED_AT'].min()}")
print(f"US latest pull: {us_df['DATA_PULLED_AT'].max()}")
us_df.head()

Found 42 PrizePicks player point props
DFS earliest pull: 2025-12-15 15:02:00
DFS latest pull: 2025-12-15 15:02:00
US earliest pull: 2025-12-15 15:03:13
US latest pull: 2025-12-15 15:03:13


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,FanDuel,player_points,Tobias Harris,Over,13.5,-108,2025-12-16,2025-12-15T23:02:48Z,2025-12-15 15:03:13
1,FanDuel,player_points,Tobias Harris,Under,13.5,-118,2025-12-16,2025-12-15T23:02:48Z,2025-12-15 15:03:13
2,FanDuel,player_points,Jalen Duren,Over,16.5,-104,2025-12-16,2025-12-15T23:02:48Z,2025-12-15 15:03:13
3,FanDuel,player_points,Jalen Duren,Under,16.5,-122,2025-12-16,2025-12-15T23:02:48Z,2025-12-15 15:03:13
4,FanDuel,player_points,Isaiah Stewart II,Over,7.5,-113,2025-12-16,2025-12-15T23:02:48Z,2025-12-15 15:03:13


In [4]:
# =============================================================================
# 2. GET PRIZEPICKS LINES AND FIND BEST MATCHING US ODDS
# =============================================================================

def american_to_implied(odds):
    """Convert American odds to implied probability."""
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def get_best_us_odds(player_name, line, side, us_points_df):
    """
    Find the best US sportsbook odds that match the PrizePicks line.
    Returns (best_odds, best_book) or (-137, None) if no match found.
    """
    player_lines = us_points_df[
        (us_points_df['NAME'] == player_name) &
        (us_points_df['LINE'] == line) &
        (us_points_df['OVER/UNDER'] == side)
    ]
    
    if player_lines.empty:
        return -137, None  # Default to -137 if no match
    
    # Find best odds (highest = least negative or most positive)
    best_idx = player_lines['ODDS'].idxmax()
    best_odds = int(player_lines.loc[best_idx, 'ODDS'])
    best_book = player_lines.loc[best_idx, 'BOOKMAKER']
    
    return best_odds, best_book

def get_market_fair_prob(player_name, line, side, us_points_df):
    """
    Get fair probability from US sportsbook consensus.
    Averages implied probabilities across books and deducts vig.
    """
    player_lines = us_points_df[
        (us_points_df['NAME'] == player_name) &
        (us_points_df['LINE'] == line) &
        (us_points_df['OVER/UNDER'] == side)
    ]
    
    if player_lines.empty:
        return None
    
    # Average implied probability across all books
    implied_probs = player_lines['ODDS'].apply(american_to_implied)
    avg_implied = implied_probs.mean()
    
    # Deduct half the vig (~2.5% for -110/-110)
    fair_prob = avg_implied - 0.025
    
    return max(0.01, min(0.99, fair_prob))

# Build PrizePicks props with best matching US odds
pp_props = []

for player in pp_points['NAME'].unique():
    player_data = pp_points[pp_points['NAME'] == player]
    
    over_line = pp_points[(pp_points['NAME'] == player) & (pp_points['OVER/UNDER'] == 'Over')]
    under_line = pp_points[(pp_points['NAME'] == player) & (pp_points['OVER/UNDER'] == 'Under')]
    
    if over_line.empty:
        continue
        
    line = over_line.iloc[0]['LINE']
    pp_odds = over_line.iloc[0]['ODDS']  # PrizePicks odds (for reference)
    
    # Find best matching US odds for each side
    best_us_odds_over, best_us_book_over = get_best_us_odds(player, line, 'Over', us_points)
    best_us_odds_under, best_us_book_under = get_best_us_odds(player, line, 'Under', us_points)
    
    # Get market fair probabilities from US books
    fair_over = get_market_fair_prob(player, line, 'Over', us_points)
    fair_under = get_market_fair_prob(player, line, 'Under', us_points)
    
    pp_props.append({
        'player': player,
        'line': line,
        'pp_odds': pp_odds,
        'best_us_odds_over': best_us_odds_over,
        'best_us_book_over': best_us_book_over,
        'best_us_odds_under': best_us_odds_under,
        'best_us_book_under': best_us_book_under,
        'fair_prob_over': fair_over,
        'fair_prob_under': fair_under,
    })

pp_props_df = pd.DataFrame(pp_props)
print(f"\nBuilt {len(pp_props_df)} PrizePicks props with best matching US odds")


Built 42 PrizePicks props with best matching US odds


In [5]:
# =============================================================================
# 3. FIT POINTS MODEL AND GET PROJECTIONS
# =============================================================================

# Load game logs
s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv')

def parse_minutes(min_str):
    if pd.isna(min_str): return 0.0
    if ':' in str(min_str):
        parts = str(min_str).split(':')
        return int(parts[0]) + int(parts[1])/60
    return float(min_str)

s26_prepped = s26.copy()
s26_prepped['minutes'] = s26_prepped['MIN'].apply(parse_minutes)
s26_prepped = s26_prepped.rename(columns={
    'PLAYER_ID': 'player_id', 'PLAYER_NAME': 'player_name',
    'FGA': 'fga', 'FG3A': 'fg3a', 'FTA': 'fta',
    'FGM': 'fgm', 'FG3M': 'fg3m', 'FTM': 'ftm',
    'PTS': 'pts', 'PLUS_MINUS': 'margin'
})

name_to_id = s26_prepped.groupby('player_name')['player_id'].first().to_dict()
name_to_team = s26_prepped.groupby('player_name')['TEAM_ABBREVIATION'].last().to_dict()

# Get team stats
league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]
team_stats = league_df.set_index('TEAM_ID')

In [6]:
# =============================================================================
# SMART USAGE ADJUSTMENT BASED ON STARTERS + POSITION + USAGE DATA
# Handles: OUT players, QUESTIONABLE players, and lineup changes
# =============================================================================
# 
# IMPORTANT: PTS_DELTA_STAR_OUT is a RAW POINTS DIFFERENCE, not a multiplier!
# Example: If player averages 15 pts with star, 18 pts without → delta = +3.0
# We need to convert this to a multiplier: 1 + (3.0 / 15.0) = 1.20
# =============================================================================

def normalize_name(name):
    """Normalize player name using nameDict for special characters."""
    return nameDict.get(name, name)

def parse_minutes(min_str):
    """Parse minutes from string format (MM:SS) or numeric."""
    if pd.isna(min_str): 
        return 0.0
    if ':' in str(min_str):
        parts = str(min_str).split(':')
        return int(parts[0]) + int(parts[1])/60
    return float(min_str)

# Build player stats cache from training data
def build_player_stats_cache(df):
    """
    Build a cache of player stats for usage adjustment calculations.
    
    Key insight: PTS_DELTA_STAR_OUT is raw points difference, not a multiplier.
    We need the player's baseline to convert it properly.
    """
    player_stats = {}
    
    for player_name in df['PLAYER_NAME'].unique():
        player_df = df[df['PLAYER_NAME'] == player_name]
        if len(player_df) < 5:
            continue
            
        # Get most recent data (last 15 games for stability)
        recent = player_df.tail(15)
        
        # Position (from binary columns)
        guard = recent['GUARD'].mean() > 0.5 if 'GUARD' in recent.columns else False
        forward = recent['FORWARD'].mean() > 0.5 if 'FORWARD' in recent.columns else False
        center = recent['CENTER'].mean() > 0.5 if 'CENTER' in recent.columns else False
        
        # Get usage rate
        usg_pct = recent['USG_PCT'].mean() if 'USG_PCT' in recent.columns else 0.20
        
        # Parse minutes properly
        if 'MIN' in recent.columns:
            mins_vals = recent['MIN'].apply(lambda x: parse_minutes(x) if pd.notna(x) else 0)
            minutes = mins_vals.mean()
        else:
            minutes = 20.0
        
        # Get average points (baseline for calculating boost multiplier)
        pts_avg = recent['PTS'].mean() if 'PTS' in recent.columns else 10.0
        
        # =================================================================
        # FIXED: Convert PTS_DELTA_STAR_OUT to a proper multiplier
        # =================================================================
        pts_boost_multiplier = 1.0  # Default: no boost
        
        if 'PTS_DELTA_STAR_OUT' in recent.columns:
            delta = recent['PTS_DELTA_STAR_OUT'].mean()
            
            if pd.notna(delta) and pts_avg > 0:
                # Convert delta to multiplier
                raw_multiplier = 1 + (delta / pts_avg)
                
                # Sanity check: cap between 0.70 and 1.40
                if 0.70 <= raw_multiplier <= 1.40:
                    pts_boost_multiplier = raw_multiplier
        
        # Also check for games without star sample size
        games_without_star = 0
        if 'GAMES_WITHOUT_STAR' in recent.columns:
            games_without_star = recent['GAMES_WITHOUT_STAR'].iloc[-1] if len(recent) > 0 else 0
        
        player_stats[player_name] = {
            'guard': guard,
            'forward': forward,
            'center': center,
            'usg_pct': usg_pct,
            'minutes': minutes,
            'pts_avg': pts_avg,
            'pts_boost_star_out': pts_boost_multiplier,
            'games_without_star': games_without_star,
            'team': recent['TEAM_ABBREVIATION'].iloc[-1] if 'TEAM_ABBREVIATION' in recent.columns else 'UNK',
            'sample_size': len(recent)
        }
    
    return player_stats

# Build the stats cache
player_stats_cache = build_player_stats_cache(s26)
print(f"Built player stats cache for {len(player_stats_cache)} players")

# Quick sanity check on boost values
boost_values = [v['pts_boost_star_out'] for v in player_stats_cache.values() if v['pts_boost_star_out'] != 1.0]
if boost_values:
    print(f"Players with star-out boost data: {len(boost_values)}")
    print(f"  Boost range: {min(boost_values):.2f}x to {max(boost_values):.2f}x")
    print(f"  Mean boost: {np.mean(boost_values):.2f}x")

# Helper functions
def get_player_position(player_name):
    """Get player position from stats cache."""
    stats = player_stats_cache.get(normalize_name(player_name), {})
    positions = []
    if stats.get('guard', False):
        positions.append('guard')
    if stats.get('forward', False):
        positions.append('forward')
    if stats.get('center', False):
        positions.append('center')
    return positions if positions else ['unknown']

def is_same_position(player1, player2):
    """Check if two players share at least one position."""
    pos1 = set(get_player_position(player1))
    pos2 = set(get_player_position(player2))
    return bool(pos1 & pos2)

def is_ball_handler(player_name):
    """Check if player is a primary ball handler (guard with decent usage)."""
    positions = get_player_position(player_name)
    usg = get_player_usage_rate(player_name)
    return 'guard' in positions and usg > 0.18

def get_player_usage_rate(player_name):
    """Get player's usage rate from stats cache."""
    stats = player_stats_cache.get(normalize_name(player_name), {})
    return stats.get('usg_pct', 0.20)

def get_player_minutes(player_name):
    """Get player's average minutes from stats cache."""
    stats = player_stats_cache.get(normalize_name(player_name), {})
    return stats.get('minutes', 20.0)

def get_player_pts_avg(player_name):
    """Get player's average points from stats cache."""
    stats = player_stats_cache.get(normalize_name(player_name), {})
    return stats.get('pts_avg', 10.0)

def get_historical_boost(player_name):
    """
    Get historical boost multiplier when star is out.
    Already converted to multiplier format (e.g., 1.15 for 15% boost).
    """
    stats = player_stats_cache.get(normalize_name(player_name), {})
    boost = stats.get('pts_boost_star_out', 1.0)
    games = stats.get('games_without_star', 0)
    return boost, games

def get_game_spread(team_abbrev, current_date, team_lines_dir='data/raw/team_lines'):
    """
    Get the spread for a team's game on a given date.
    """
    # Convert date to file format (YYYYMMDD)
    date_str = current_date.replace('-', '')
    
    # Find all files matching the date pattern
    team_lines_path = Path(team_lines_dir)
    pattern = f'NBA_{date_str}_*.json'
    matching_files = list(team_lines_path.glob(pattern))
    
    if not matching_files:
        return None
    
    # Get the latest file (by modification time, or by time in filename)
    # Option 1: By modification time (most recent scrape)
    latest_file = max(matching_files, key=lambda p: p.stat().st_mtime)
    
    # Option 2: By time in filename (if you prefer)
    # latest_file = max(matching_files, key=lambda p: int(p.stem.split('_')[-1]) if p.stem.split('_')[-1].isdigit() else 0)
    
    try:
        with open(latest_file, 'r') as f:
            games_data = json.load(f)
        
        # Map team abbreviations to full names
        team_name_map = {
            'ATL': 'Atlanta Hawks', 'BOS': 'Boston Celtics', 'BKN': 'Brooklyn Nets',
            'CHA': 'Charlotte Hornets', 'CHI': 'Chicago Bulls', 'CLE': 'Cleveland Cavaliers',
            'DAL': 'Dallas Mavericks', 'DEN': 'Denver Nuggets', 'DET': 'Detroit Pistons',
            'GSW': 'Golden State Warriors', 'HOU': 'Houston Rockets', 'IND': 'Indiana Pacers',
            'LAC': 'LA Clippers', 'LAL': 'Los Angeles Lakers', 'MEM': 'Memphis Grizzlies',
            'MIA': 'Miami Heat', 'MIL': 'Milwaukee Bucks', 'MIN': 'Minnesota Timberwolves',
            'NOP': 'New Orleans Pelicans', 'NYK': 'New York Knicks', 'OKC': 'Oklahoma City Thunder',
            'ORL': 'Orlando Magic', 'PHI': 'Philadelphia 76ers', 'PHX': 'Phoenix Suns',
            'POR': 'Portland Trail Blazers', 'SAC': 'Sacramento Kings', 'SAS': 'San Antonio Spurs',
            'TOR': 'Toronto Raptors', 'UTA': 'Utah Jazz', 'WAS': 'Washington Wizards'
        }
        
        team_full_name = team_name_map.get(team_abbrev)
        if not team_full_name:
            return None
        
        # Find the game with this team
        for game in games_data:
            is_home = game['home_team'] == team_full_name
            is_away = game['away_team'] == team_full_name
            
            if is_home or is_away:
                # Get spread from first bookmaker (or average across bookmakers)
                for bookmaker in game['bookmakers']:
                    for market in bookmaker['markets']:
                        if market['market_key'] == 'spreads':
                            for outcome in market['outcomes']:
                                if outcome['name'] == team_full_name:
                                    spread = outcome['point']
                                    # Spread is from team's perspective
                                    # Negative = favored (expected to win by that amount)
                                    # Positive = underdog (expected to lose by that amount)
                                    return spread
        return None
    except Exception as e:
        return None

def calculate_blowout_prob_from_spread(spread):
    """
    Calculate blowout probability from spread.
    
    Blowout = |actual margin| > 20
    Using spread as expected margin, calculate probability of blowout.
    """
    if spread is None:
        return 0.15  # Default fallback
    
    # Expected margin from team's perspective
    # If spread is -4.5, team is expected to win by 4.5
    # If spread is +4.5, team is expected to lose by 4.5
    expected_margin = -spread  # Flip sign: negative spread = positive margin
    
    # Typical NBA game margin std dev is ~12 points
    margin_std = 12.0
    
    # Probability of blowout = P(|margin| > 20)
    # This is P(margin > 20) + P(margin < -20)
    prob_win_blowout = 1 - stats.norm.cdf(20, expected_margin, margin_std)
    prob_loss_blowout = stats.norm.cdf(-20, expected_margin, margin_std)
    blowout_prob = prob_win_blowout + prob_loss_blowout
    
    # Clamp between reasonable bounds
    return max(0.05, min(0.40, blowout_prob))

def get_out_and_questionable_players(team_abbrev):
    """
    Get lists of out and questionable players for a team.
    Assumes outPlayers and questionablePlayers dicts are available globally.
    
    Returns:
        tuple: (list of out players, list of questionable players)
    """
    out = outPlayers.get(team_abbrev, [])
    questionable = questionablePlayers.get(team_abbrev, [])
    return out, questionable

def get_usage_adjustment(player_name, team_abbrev):
    """
    Smart usage adjustment based on:
    1. OUT players (100% certainty they won't play)
    2. QUESTIONABLE players (50% probability weighted adjustment)
    3. Projected vs main lineup changes
    4. Historical performance when stars out
    5. Position overlap for usage redistribution
    6. Starter status changes
    
    Returns:
        tuple: (adjustment_multiplier, reason_string)
    """
    adjustment = 1.0
    reasons = []
    
    # Normalize player name
    player_name_normalized = normalize_name(player_name)
    
    # Get lineups
    main_lineup = mainStartingFive.get(team_abbrev, [])
    projected = projectedStartingFive.get(team_abbrev, [])
    team_star = teamStarPlayer.get(team_abbrev)
    
    # Get out and questionable players
    out_players_raw, questionable_players_raw = get_out_and_questionable_players(team_abbrev)
    
    # Normalize names for comparison
    main_normalized = [normalize_name(p) for p in main_lineup]
    projected_normalized = [normalize_name(p) for p in projected]
    out_normalized = [normalize_name(p) for p in out_players_raw]
    questionable_normalized = [normalize_name(p) for p in questionable_players_raw]
    
    # =========================================================================
    # CHECK IF PLAYER IS OUT OR QUESTIONABLE
    # =========================================================================
    if player_name_normalized in out_normalized:
        return 0.0, "Player is OUT - do not bet"
    
    if player_name_normalized in questionable_normalized:
        # Questionable players get reduced projection
        adjustment *= 0.75
        reasons.append("Player is QUESTIONABLE: -25%")
    
    # =========================================================================
    # Find who's definitely out (confirmed + not in projected lineup)
    # =========================================================================
    confirmed_out = []
    
    # Players on injury report as OUT
    for player in out_players_raw:
        if normalize_name(player) not in projected_normalized:
            confirmed_out.append(player)
    
    # Players in main lineup but not in projected (and not questionable)
    for player in main_lineup:
        player_norm = normalize_name(player)
        if (player_norm not in projected_normalized and 
            player_norm not in out_normalized and
            player_norm not in questionable_normalized):
            confirmed_out.append(player)
    
    # Check player's starting status
    player_in_main = player_name_normalized in main_normalized
    player_in_projected = player_name_normalized in projected_normalized
    
    # =========================================================================
    # CASE 1: No one is out - only check starter status
    # =========================================================================
    if not confirmed_out and not questionable_players_raw:
        if player_in_main and not player_in_projected:
            # Player usually starts but NOT starting tonight
            adjustment *= 0.85
            reasons.append("Not starting tonight: -15%")
        elif player_in_projected and not player_in_main:
            # Player starting tonight but usually doesn't
            adjustment *= 1.08
            reasons.append("Starting tonight (expanded): +8%")
        
        adjustment = min(1.30, max(0.70, adjustment))
        return adjustment, "; ".join(reasons) if reasons else "Full strength"
    
    # =========================================================================
    # CASE 2: Someone is out or questionable - calculate boost
    # =========================================================================
    
    # Check if star player is affected
    star_is_out = team_star and normalize_name(team_star) in [normalize_name(p) for p in confirmed_out]
    star_is_questionable = team_star and normalize_name(team_star) in questionable_normalized
    
    # Check for historical data (only use if star is definitely out)
    historical_boost, games_sample = get_historical_boost(player_name)
    has_reliable_historical = (
        games_sample >= 3 and  # At least 3 games without star
        0.85 <= historical_boost <= 1.35 and  # Reasonable range
        historical_boost != 1.0  # Actually has data
    )
    
    if has_reliable_historical and star_is_out:
        # Use historical data with regression toward 1.0
        sample_weight = min(0.8, games_sample / 10)  # Max 80% weight on historical
        regressed_boost = historical_boost * sample_weight + 1.0 * (1 - sample_weight)
        
        adjustment *= regressed_boost
        boost_pct = (historical_boost - 1) * 100
        reasons.append(f"Star out history ({games_sample}g): {boost_pct:+.0f}% → {(regressed_boost-1)*100:+.0f}% regressed")
    
    elif has_reliable_historical and star_is_questionable:
        # Star is questionable: apply 50% of the historical boost
        sample_weight = min(0.8, games_sample / 10)
        regressed_boost = historical_boost * sample_weight + 1.0 * (1 - sample_weight)
        partial_boost = 1.0 + (regressed_boost - 1.0) * 0.5  # 50% of boost
        
        adjustment *= partial_boost
        boost_pct = (historical_boost - 1) * 100
        reasons.append(f"Star questionable ({games_sample}g history): {boost_pct:+.0f}% → {(partial_boost-1)*100:+.0f}% (50% weighted)")
    
    else:
        # No reliable historical data - estimate from usage redistribution
        
        # Process CONFIRMED OUT players (100% weight)
        for out_player in confirmed_out:
            if normalize_name(out_player) == player_name_normalized:
                continue
            out_usage = get_player_usage_rate(out_player)

            # Determine capture rate based on relationship
            if is_same_position(player_name, out_player):
                capture_rate = 0.30  # Same position captures most
                reason_tag = "same pos"
            elif is_ball_handler(player_name) and is_ball_handler(out_player):
                capture_rate = 0.25
                reason_tag = "ball handler"
            elif player_in_projected:
                capture_rate = 0.12  # Starters get indirect boost
                reason_tag = "starter"
            else:
                capture_rate = 0.05  # Bench minimal
                reason_tag = "bench"
            
            # Convert to boost multiplier
            player_usage = get_player_usage_rate(player_name)
            if player_usage > 0.05:
                usage_gained = out_usage * capture_rate
                boost_pct = usage_gained / player_usage
                boost_multiplier = 1 + min(0.18, boost_pct)  # Cap at 18% from any single player
                
                if boost_multiplier > 1.02:
                    adjustment *= boost_multiplier
                    reasons.append(f"{out_player} OUT ({reason_tag}): +{(boost_multiplier-1)*100:.0f}%")
        
        # Process QUESTIONABLE players (50% weight - they might play)
        for q_player in questionable_players_raw:
            q_player_norm = normalize_name(q_player)
            
            # Skip if player already in confirmed out list
            # Skip if this is the player themselves (they can't get a boost from being questionable)
            if q_player_norm == player_name_normalized:
                continue
            
            # Skip if player already in confirmed out list
            if q_player_norm in [normalize_name(p) for p in confirmed_out]:
                continue

            q_usage = get_player_usage_rate(q_player)
            
            # Determine capture rate (same logic as above)
            if is_same_position(player_name, q_player):
                capture_rate = 0.30
                reason_tag = "same pos"
            elif is_ball_handler(player_name) and is_ball_handler(q_player):
                capture_rate = 0.25
                reason_tag = "ball handler"
            elif player_in_projected:
                capture_rate = 0.12
                reason_tag = "starter"
            else:
                capture_rate = 0.05
                reason_tag = "bench"
            
            # Apply 50% probability weight for questionable status
            player_usage = get_player_usage_rate(player_name)
            if player_usage > 0.05:
                usage_gained = q_usage * capture_rate * 0.5  # 50% weight
                boost_pct = usage_gained / player_usage
                boost_multiplier = 1 + min(0.09, boost_pct)  # Cap at 9% (half of 18%)
                
                if boost_multiplier > 1.01:
                    adjustment *= boost_multiplier
                    reasons.append(f"{q_player} QUESTIONABLE ({reason_tag}): +{(boost_multiplier-1)*100:.0f}% (50% weighted)")
    
    # =========================================================================
    # Starter status adjustments (apply on top of above)
    # =========================================================================
    if player_in_main and not player_in_projected:
        adjustment *= 0.85
        reasons.append("Not starting tonight: -15%")
    elif player_in_projected and not player_in_main:
        adjustment *= 1.05
        reasons.append("Expanded role: +5%")
    
    # Final cap
    adjustment = min(1.30, max(0.0, adjustment))  # Allow 0.0 for OUT players
    
    return adjustment, "; ".join(reasons) if reasons else "No adjustment"

# =============================================================================
# Calculate usage adjustments for all players
# =============================================================================
usage_adjustments = {}
print("\n" + "="*80)
print("SMART USAGE ADJUSTMENTS (OUT + QUESTIONABLE PLAYERS)")
print("="*80)

# Show injury report summary
total_out = sum(len(players) for players in outPlayers.values())
total_questionable = sum(len(players) for players in questionablePlayers.values())
print(f"\n📋 Injury Report: {total_out} OUT, {total_questionable} QUESTIONABLE")

if total_out > 0:
    print("\n🚫 OUT Players:")
    for team, players in outPlayers.items():
        if players:
            print(f"  {team}: {', '.join(players)}")

if total_questionable > 0:
    print("\n❓ QUESTIONABLE Players:")
    for team, players in questionablePlayers.items():
        if players:
            print(f"  {team}: {', '.join(players)}")

print("\n" + "="*80)

adjusted_players = []
out_player_list = []

for player in pp_props_df['player'].unique():
    team = name_to_team.get(player, 'UNK')
    if team == 'UNK':
        usage_adjustments[player] = (1.0, "Unknown team")
        continue
        
    adj, reason = get_usage_adjustment(player, team)
    usage_adjustments[player] = (adj, reason)
    
    # Track players who are out
    if adj == 0.0:
        out_player_list.append({
            'player': player,
            'team': team,
            'reason': reason
        })
    elif adj != 1.0 and reason not in ["Full strength", "No adjustment"]:
        adjusted_players.append({
            'player': player,
            'team': team,
            'adjustment': adj,
            'reason': reason,
            'position': "/".join(get_player_position(player)),
            'usg_pct': get_player_usage_rate(player),
            'pts_avg': get_player_pts_avg(player)
        })

# Summary
boosts = [p for p in adjusted_players if p['adjustment'] > 1.0]
reductions = [p for p in adjusted_players if p['adjustment'] < 1.0]

print(f"\n📊 Summary: {len(out_player_list)} OUT, {len(boosts)} boosted, {len(reductions)} reduced, {len(pp_props_df['player'].unique()) - len(adjusted_players) - len(out_player_list)} unchanged")

if out_player_list:
    print(f"\n🚫 PLAYERS OUT (DO NOT BET) - {len(out_player_list)}:")
    for p in out_player_list:
        print(f"  ✗ {p['player']} ({p['team']}) - {p['reason']}")

if boosts:
    print(f"\n📈 BOOSTED ({len(boosts)}):")
    for p in sorted(boosts, key=lambda x: x['adjustment'], reverse=True)[:15]:
        print(f"  ↑ {p['player']} ({p['team']}) {p['adjustment']:.2f}x")
        print(f"     {p['position']}, USG:{p['usg_pct']:.1%}, Avg:{p['pts_avg']:.1f}pts")
        print(f"     {p['reason']}")

if reductions:
    print(f"\n📉 REDUCED ({len(reductions)}):")
    for p in sorted(reductions, key=lambda x: x['adjustment'])[:10]:
        print(f"  ↓ {p['player']} ({p['team']}) {p['adjustment']:.2f}x")
        print(f"     {p['reason']}")

Built player stats cache for 448 players
Players with star-out boost data: 194
  Boost range: 0.70x to 1.40x
  Mean boost: 1.03x

SMART USAGE ADJUSTMENTS (OUT + QUESTIONABLE PLAYERS)

📋 Injury Report: 19 OUT, 7 QUESTIONABLE

🚫 OUT Players:
  BOS: Jayson Tatum
  TOR: RJ Barrett
  MIA: Pelle Larsson, Terry Rozier
  DAL: D'Angelo Russell, Kyrie Irving
  UTA: Jusuf Nurkić, Georges Niang
  HOU: Dorian Finney-Smith, Fred VanVleet
  DEN: Aaron Gordon, Christian Braun
  MEM: Ty Jerome, Scotty Pippen, Zach Edey, John Konchar, Javon Small
  LAC: Chris Paul, Derrick Jones

❓ QUESTIONABLE Players:
  DAL: Daniel Gafford, Anthony Davis, Brandon Williams
  DEN: Julian Strawther
  HOU: Tari Eason
  LAC: Nicolas Batum, James Harden


📊 Summary: 0 OUT, 34 boosted, 2 reduced, 6 unchanged

📈 BOOSTED (34):
  ↑ Tyler Herro (MIA) 1.30x
     guard, USG:24.8%, Avg:23.2pts
     Pelle Larsson OUT (same pos): +18%; Terry Rozier OUT (starter): +10%; Expanded role: +5%
  ↑ Kyle Filipowski (UTA) 1.30x
     center, U

In [7]:
# =============================================================================
# Fit model and get projections
# =============================================================================
model = PointsPropModel(min_edge=0.02, min_confidence=0.52)
model.fit(s26_prepped)

# Create mapping from team abbreviation to team_id
# Use s26_prepped which has both TEAM_ABBREVIATION and TEAM_ID
team_abbrev_to_id = s26_prepped.groupby('TEAM_ABBREVIATION')['TEAM_ID'].first().to_dict()

player_projections = {}

# Then in your projection loop (around line 997-1012):
for _, row in pp_props_df.iterrows():
    player = row['player']
    player_id = name_to_id.get(player)
    
    if player_id is None:
        continue
    
    usage_adj, adj_reason = usage_adjustments.get(player, (1.0, "No adjustment"))
    
    # Skip players who are OUT
    if usage_adj == 0.0:
        continue
    
    # Calculate if it's a back-to-back game
    player_games = s26_prepped[s26_prepped['player_id'] == player_id].sort_values('GAME_DATE')
    is_b2b = False
    if not player_games.empty:
        latest_game_date = pd.to_datetime(player_games['GAME_DATE'].iloc[-1])
        current_date_dt = pd.to_datetime(current_date)
        days_since_last_game = (current_date_dt - latest_game_date).days
        is_b2b = (days_since_last_game == 1)
    
    # Calculate blowout probability from spread
    player_team_abbrev = name_to_team.get(player)
    spread = get_game_spread(player_team_abbrev, current_date) if player_team_abbrev else None
    blowout_prob = calculate_blowout_prob_from_spread(spread)
    
    # Get opponent team abbreviation using findOpp
    opp_team_id = None
    try:
        opp_abbrev, _ = findOpp(player, s26, current_date)
        if opp_abbrev and opp_abbrev in team_abbrev_to_id:
            opp_team_id = int(team_abbrev_to_id[opp_abbrev])
    except Exception as e:
        pass
        
    try:
        projection = model.project_points(
            player_id=player_id,
            is_b2b=is_b2b,
            blowout_prob=blowout_prob, 
            usage_adjustment=usage_adj,
            opp_team_id=opp_team_id
        )
        if projection:
            player_projections[player] = {
                'expected': projection['expected_points'],
                'std': projection['std'],
                'team': name_to_team.get(player, 'UNK'),
                'usage_adj': usage_adj,
                'usage_reason': adj_reason
            }
    except Exception as e:
        continue

In [8]:
# =============================================================================
# 4. CALCULATE SINGLE LEG EDGES (USING BEST US ODDS FOR EV) - FIXED VERSION
# =============================================================================

def american_to_implied(odds):
    """Convert American odds to implied probability."""
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def evaluate_prop_with_volatility(model, player_id, market_line, market_juice, 
                                   player_logs, **projection_kwargs):
    """
    Enhanced prop evaluation that incorporates volatility analysis.
    
    Uses better distribution assumptions (empirical/Student's t) instead of normal,
    and incorporates volatility-based edge adjustments.
    """
    # Get base evaluation
    base_eval = model.evaluate_prop(
        player_id=player_id,
        market_line=market_line,
        market_juice=market_juice,
        **projection_kwargs
    )
    
    if not base_eval:
        return None
    
    # Get volatility analysis (requires at least 15 games for meaningful analysis)
    if len(player_logs) < 15:
        # Not enough data for volatility analysis, use base evaluation
        return base_eval
    
    vol_analysis = model.analyze_volatility(player_id)
    
    if not vol_analysis:
        # Volatility analysis failed, use base evaluation
        return base_eval
    
    # Get true probability using better distribution assumptions
    true_prob = model.volatility_analyzer.calculate_true_probability(
        player_logs,
        line=market_line,
        stat_col='pts',
        use_empirical=True
    )
    
    # Use empirical probability if available (most accurate with enough data)
    # Otherwise fall back to Student's t (fatter tails than normal)
    if true_prob['sample_size'] >= 20:
        enhanced_prob_over = true_prob['empirical']['over']
        enhanced_prob_under = true_prob['empirical']['under']
    else:
        enhanced_prob_over = true_prob['t_dist']['over']
        enhanced_prob_under = true_prob['t_dist']['under']
    
    # Get base edges
    base_edge_over = base_eval['edge_analysis']['over_edge']
    base_edge_under = base_eval['edge_analysis']['under_edge']
    
    # Calculate enhanced edges using better probabilities
    market_implied = american_to_implied(market_juice)
    enhanced_edge_over = enhanced_prob_over - market_implied
    enhanced_edge_under = enhanced_prob_under - (1 - market_implied)
    
    # Add volatility-based adjustments
    mins_vol = vol_analysis['minutes_volatility']
    recency = vol_analysis['recency_bias']
    
    # Minutes volatility adjustment (conservative: 20% of detected edge)
    if mins_vol.get('edge_magnitude', 0) > 0:
        volatility_adj = mins_vol['edge_magnitude'] * 0.2
        enhanced_edge_over += volatility_adj
        enhanced_edge_under += volatility_adj
    
    # Recency bias adjustment
    if recency.get('edge_direction'):
        recency_adj = recency['edge_magnitude']
        if recency['edge_direction'] == 'OVER':
            enhanced_edge_over += recency_adj
        elif recency['edge_direction'] == 'UNDER':
            enhanced_edge_under += recency_adj
    
    # Distribution tail analysis (fat tails = more extreme outcomes)
    dist = vol_analysis.get('distribution', {})
    if dist.get('tail_analysis'):
        tail = dist['tail_analysis']
        # If fat tails detected, small adjustment for extreme outcomes
        if tail.get('fat_upper_tail'):
            enhanced_edge_over += 0.015  # Small boost for over (more extreme highs)
        if tail.get('fat_lower_tail'):
            enhanced_edge_under += 0.015  # Small boost for under (more extreme lows)
    
    # Determine best recommendation
    best_edge = max(enhanced_edge_over, enhanced_edge_under)
    if best_edge == enhanced_edge_over:
        recommendation = 'OVER'
        best_prob = enhanced_prob_over
    else:
        recommendation = 'UNDER'
        best_prob = enhanced_prob_under
    
    # Calculate expected value
    if recommendation == 'OVER':
        ev = enhanced_edge_over * (1 - market_implied)
    else:
        ev = enhanced_edge_under * market_implied
    
    # Update edge analysis with enhanced values
    enhanced_eval = base_eval.copy()
    enhanced_eval['edge_analysis']['prob_over'] = enhanced_prob_over
    enhanced_eval['edge_analysis']['prob_under'] = enhanced_prob_under
    enhanced_eval['edge_analysis']['over_edge'] = enhanced_edge_over
    enhanced_eval['edge_analysis']['under_edge'] = enhanced_edge_under
    enhanced_eval['edge_analysis']['recommendation'] = recommendation
    enhanced_eval['edge_analysis']['expected_value'] = ev
    enhanced_eval['edge_analysis']['confidence'] = best_prob
    
    # Store volatility analysis for reference
    enhanced_eval['volatility_analysis'] = {
        'distribution_fit': dist.get('best_fit', 'normal'),
        'kurtosis': dist.get('kurtosis', 0),
        'skewness': dist.get('skewness', 0),
        'minutes_volatility_edge': mins_vol.get('edge_magnitude', 0),
        'recency_bias_edge': recency.get('edge_magnitude', 0),
        'edge_vs_normal': true_prob.get('edge_vs_normal', {})
    }
    
    return enhanced_eval

def calculate_hit_rate(player_name, line, side, data_df, windows=[5, 10, 15]):
    """
    Calculate hit rate percentage for a player hitting a line (over/under) in last N games.
    
    Args:
        player_name: Player name
        line: The line value
        side: 'Over' or 'Under'
        data_df: DataFrame with historical game data (should have PLAYER_NAME, PTS, GAME_DATE)
        windows: List of game windows to calculate (default: [5, 10, 15])
    
    Returns:
        Dictionary with hit rates for each window (as percentages)
    """
    # Filter to player's games
    player_df = data_df[data_df['PLAYER_NAME'] == player_name].copy()
    
    if len(player_df) == 0:
        return {f'L-{w}': None for w in windows}
    
    # Sort by date (most recent last)
    player_df = player_df.sort_values('GAME_DATE')
    
    results = {}
    
    for window in windows:
        # Get last N games
        if len(player_df) < window:
            last_n_games = player_df
            actual_window = len(player_df)
        else:
            last_n_games = player_df.tail(window)
            actual_window = window
        
        if actual_window == 0:
            results[f'L-{window}'] = None
            continue
        
        # Calculate hits based on side
        if side.lower() == 'over':
            hits = (last_n_games['PTS'] > line).sum()
        else:  # under
            hits = (last_n_games['PTS'] < line).sum()
        
        # Calculate percentage
        hit_rate_pct = (hits / actual_window) * 100
        results[f'L-{window}'] = round(hit_rate_pct, 1)
    
    return results

single_bets = []

for _, row in pp_props_df.iterrows():
    player = row['player']
    line = row['line']
    pp_odds = row['pp_odds']
    
    # Get player_id for evaluate_prop
    player_id = name_to_id.get(player)
    if player_id is None:
        continue
    
    # Get usage adjustment
    usage_adj, adj_reason = usage_adjustments.get(player, (1.0, "No adjustment"))
    
    # Skip players who are OUT
    if usage_adj == 0.0:
        continue
    
    # Calculate if it's a back-to-back game
    player_games = s26_prepped[s26_prepped['player_id'] == player_id].sort_values('GAME_DATE')
    is_b2b = False
    if not player_games.empty:
        latest_game_date = pd.to_datetime(player_games['GAME_DATE'].iloc[-1])
        current_date_dt = pd.to_datetime(current_date)
        days_since_last_game = (current_date_dt - latest_game_date).days
        is_b2b = (days_since_last_game == 1)
    
    # Calculate blowout probability from spread
    player_team_abbrev = name_to_team.get(player)
    spread = get_game_spread(player_team_abbrev, current_date) if player_team_abbrev else None
    blowout_prob = calculate_blowout_prob_from_spread(spread)
    
    # Get opponent team ID
    opp_team_id = None
    try:
        opp_abbrev, _ = findOpp(player, s26, current_date)
        if opp_abbrev and opp_abbrev in team_abbrev_to_id:
            opp_team_id = int(team_abbrev_to_id[opp_abbrev])
    except Exception:
        pass
    
    # Determine best US odds to use (for edge calculation)
    # Check both sides to find best odds
    best_us_odds_over = row.get('best_us_odds_over', pp_odds)
    best_us_odds_under = row.get('best_us_odds_under', pp_odds)
    best_us_book_over = row.get('best_us_book_over', 'PrizePicks')
    best_us_book_under = row.get('best_us_book_under', 'PrizePicks')
    
    # Market fair probabilities (from US books)
    fair_over = row.get('fair_prob_over', 0.5) or 0.5
    fair_under = row.get('fair_prob_under', 0.5) or 0.5
    
    # PrizePicks implied probability
    pp_implied = american_to_implied(pp_odds)
    
    # ✅ USE evaluate_prop_with_volatility() FOR BOTH SIDES TO FIND BEST EDGE
    # Get player logs for volatility analysis
    player_logs = s26_prepped[s26_prepped['player_id'] == player_id].sort_values('GAME_DATE')
    
    evaluations = {}
    
    # Evaluate Over side with volatility analysis
    try:
        eval_over = evaluate_prop_with_volatility(
            model=model,
            player_id=player_id,
            market_line=line,
            market_juice=int(best_us_odds_over),
            player_logs=player_logs,
            is_b2b=is_b2b,
            blowout_prob=blowout_prob,
            usage_adjustment=usage_adj,
            opp_team_id=opp_team_id
        )
        if eval_over:
            evaluations['over'] = eval_over
    except Exception:
        pass
    
    # Evaluate Under side with volatility analysis
    try:
        eval_under = evaluate_prop_with_volatility(
            model=model,
            player_id=player_id,
            market_line=line,
            market_juice=int(best_us_odds_under),
            player_logs=player_logs,
            is_b2b=is_b2b,
            blowout_prob=blowout_prob,
            usage_adjustment=usage_adj,
            opp_team_id=opp_team_id
        )
        if eval_under:
            evaluations['under'] = eval_under
    except Exception:
        pass
    
    if not evaluations:
        continue
    
    # Determine best side based on edge vs fair market
    edge_over = evaluations['over']['edge_analysis']['over_edge'] if 'over' in evaluations else -999
    edge_under = evaluations['under']['edge_analysis']['under_edge'] if 'under' in evaluations else -999
    
    # Also calculate edge vs PrizePicks (50% breakeven)
    if 'over' in evaluations:
        model_prob_over = evaluations['over']['edge_analysis']['prob_over']
        edge_vs_pp_over = model_prob_over - 0.50
    else:
        model_prob_over = 0.5
        edge_vs_pp_over = 0.0
    
    if 'under' in evaluations:
        model_prob_under = evaluations['under']['edge_analysis']['prob_under']
        edge_vs_pp_under = model_prob_under - 0.50
    else:
        model_prob_under = 0.5
        edge_vs_pp_under = 0.0
    
    # 🔧 FIX: Determine side based on PROJECTION vs LINE, not edge magnitude
    # Get the projection to determine natural side
    if 'over' in evaluations:
        projection_value = evaluations['over']['projection']['expected_points']
    elif 'under' in evaluations:
        projection_value = evaluations['under']['projection']['expected_points']
    else:
        continue  # Should never happen since we checked earlier
    
    # Select side based on projection
    if projection_value > line:
        # Project over, so take OVER
        best_side = 'Over'
        if 'over' not in evaluations:
            continue  # Skip if we don't have over evaluation
        best_eval = evaluations['over']
        best_model_prob = model_prob_over
        best_fair_prob = fair_over
        best_edge = edge_over
        best_edge_vs_pp = edge_vs_pp_over
        best_us_odds = best_us_odds_over
        best_us_book = best_us_book_over
    else:
        # Project under, so take UNDER
        best_side = 'Under'
        if 'under' not in evaluations:
            continue  # Skip if we don't have under evaluation
        best_eval = evaluations['under']
        best_model_prob = model_prob_under
        best_fair_prob = fair_under
        best_edge = edge_under
        best_edge_vs_pp = edge_vs_pp_under
        best_us_odds = best_us_odds_under
        best_us_book = best_us_book_under
    
    # Handle missing US odds
    if pd.isna(best_us_odds) or best_us_odds is None:
        best_us_odds = pp_odds
        best_us_book = f'Default ({pp_odds})'
    
    # Extract values from evaluation
    projection = best_eval['projection']
    edge_analysis = best_eval['edge_analysis']
    
    # 🔧 FIX: Get the base projection value
    base_projection = projection['expected_points']
    
    # 🔧 FIX: Adjust projection display based on side selected
    # For UNDER picks, show how far below the line we project
    # For OVER picks, show the actual projection
    if best_side == 'Under':
        # Show projection as-is (it should already be below line for under picks)
        display_projection = base_projection
        # If projection is above line but we're picking under, it means the edge comes from probability distribution
        # In this case, we should note this in confidence assessment
        if base_projection > line:
            # Model projects above line but probability analysis favors under
            # This can happen with fat-tailed distributions or volatility adjustments
            pts_diff_for_confidence = abs(base_projection - line)
        else:
            pts_diff_for_confidence = abs(line - base_projection)
    else:  # Over
        display_projection = base_projection
        pts_diff_for_confidence = abs(base_projection - line)
    
    # 🔧 FIX: Calculate correct EV and Kelly based on the selected side
    if best_side == 'Over':
        # For Over bets: EV = (model_prob * (odds_decimal - 1)) - ((1 - model_prob) * 1)
        odds_decimal = (100 / abs(best_us_odds)) + 1 if best_us_odds < 0 else (best_us_odds / 100) + 1
        ev = (best_model_prob * (odds_decimal - 1)) - ((1 - best_model_prob) * 1)
        
        # Kelly for Over
        if best_edge > 0 and best_us_odds != 0:
            # Kelly = (bp - q) / b, where b = decimal odds - 1, p = win prob, q = loss prob
            b = odds_decimal - 1
            kelly_full = (b * best_model_prob - (1 - best_model_prob)) / b
            kelly_full = max(0, kelly_full)  # Don't go negative
        else:
            kelly_full = 0
    else:  # Under
        # For Under bets: same calculation
        odds_decimal = (100 / abs(best_us_odds)) + 1 if best_us_odds < 0 else (best_us_odds / 100) + 1
        ev = (best_model_prob * (odds_decimal - 1)) - ((1 - best_model_prob) * 1)
        
        # Kelly for Under
        if best_edge > 0 and best_us_odds != 0:
            b = odds_decimal - 1
            kelly_full = (b * best_model_prob - (1 - best_model_prob)) / b
            kelly_full = max(0, kelly_full)  # Don't go negative
        else:
            kelly_full = 0
    
    kelly_quarter = kelly_full * 0.25  # Quarter Kelly
    
    # Confidence flag based on projection vs line gap
    confidence = 'HIGH' if pts_diff_for_confidence > 1.5 * projection['std'] else 'MEDIUM' if pts_diff_for_confidence > projection['std'] else 'LOW'
    
    # 🔧 FIX: Calculate hit rates for the correct side
    hit_rates = calculate_hit_rate(player, line, best_side, s26)
    
    single_bets.append({
        'player': player,
        'team': name_to_team.get(player, 'UNK'),
        'line': line,
        'side': best_side,
        'projection': round(display_projection, 1),
        'std': round(projection['std'], 1),
        'model_prob': round(best_model_prob, 3),
        'fair_prob': round(best_fair_prob, 3) if best_fair_prob else None,
        'pp_odds': pp_odds,
        'pp_implied': round(pp_implied, 3),
        'best_us_odds': int(best_us_odds) if not pd.isna(best_us_odds) else pp_odds,
        'best_us_book': best_us_book if best_us_book else f'Default ({pp_odds})',
        'edge_vs_fair': round(best_edge, 4) if best_fair_prob else None,
        'edge_vs_pp': round(best_edge_vs_pp, 4),
        'ev': round(ev, 4),
        'ev_percent': round(ev * 100, 2),
        'kelly_quarter': round(kelly_quarter, 4),
        'confidence': confidence,
        'edge_over': round(edge_analysis['over_edge'], 4),
        'edge_under': round(edge_analysis['under_edge'], 4),
        'usage_adj': usage_adj,
        'usage_reason': adj_reason,
        'L-5': hit_rates.get('L-5'),
        'L-10': hit_rates.get('L-10'),
        'L-15': hit_rates.get('L-15'),
    })

singles_df = pd.DataFrame(single_bets)

# Sort by edge and filter
singles_df = singles_df.sort_values('edge_vs_fair', ascending=False, na_position='last')

# Display top picks
print("\n" + "="*120)
print("TOP SINGLE UNDERDOG PICKS (sorted by edge vs market fair value)")
print("="*120)
print(f"{'Player':<25} {'Team':<6} {'Side':<6} {'Line':<6} {'Proj':<7} {'Model%':<8} {'Fair%':<8} {'Edge%':<8} {'US Odds':<18} {'EV%':<8} {'Kelly':<8} {'Conf':<6}")
print("-"*120)

for _, bet in singles_df.head(20).iterrows():
    fair_str = f"{bet['fair_prob']*100:.1f}%" if bet['fair_prob'] else "N/A"
    edge_str = f"{bet['edge_vs_fair']*100:.1f}%" if bet['edge_vs_fair'] else f"{bet['edge_vs_pp']*100:.1f}%*"
    us_odds_str = f"{bet['best_us_odds']:+d}" if bet['best_us_odds'] >= 0 else f"{bet['best_us_odds']}"
    
    print(f"{bet['player']:<25} {bet['team']:<6} {bet['side']:<6} {bet['line']:<6.1f} {bet['projection']:<7.1f} "
          f"{bet['model_prob']*100:<8.1f} {fair_str:<8} {edge_str:<8} {us_odds_str:<6} ({bet['best_us_book']:<10}) "
          f"{bet['ev_percent']:<8.1f} {bet['kelly_quarter']*100:<8.2f} {bet['confidence']:<6}")

# Show usage-adjusted players separately
adjusted_bets = singles_df[singles_df['usage_adj'] != 1.0]
if len(adjusted_bets) > 0:
    print("\n" + "="*80)
    print("USAGE-ADJUSTED PLAYERS IN TOP PICKS")
    print("="*80)
    for _, bet in adjusted_bets.head(10).iterrows():
        direction = "↑" if bet['usage_adj'] > 1 else "↓"
        print(f"{direction} {bet['player']} ({bet['team']}): {bet['usage_adj']:.2f}x")
        print(f"   Reason: {bet['usage_reason']}")
        print(f"   Pick: {bet['side']} {bet['line']} (Proj: {bet['projection']}, Edge: {bet['edge_vs_pp']*100:.1f}%)")


TOP SINGLE UNDERDOG PICKS (sorted by edge vs market fair value)
Player                    Team   Side   Line   Proj    Model%   Fair%    Edge%    US Odds            EV%      Kelly    Conf  
------------------------------------------------------------------------------------------------------------------------
Ja Morant                 MEM    Over   19.5   24.9    82.2     49.8%    31.0%    -105   (BetMGM    ) 60.5     15.89    LOW   
Kyle Filipowski           UTA    Under  12.5   10.9    78.3     50.5%    29.4%    -103   (BetRivers ) 54.2     13.97    LOW   
Tyler Herro               MIA    Over   21.5   25.5    80.5     51.4%    27.7%    -112   (BetRivers ) 52.4     14.67    LOW   
Anthony Davis             DAL    Under  24.5   19.9    80.2     52.7%    26.8%    -115   (Bovada    ) 50.0     14.38    LOW   
Cameron Johnson           DEN    Under  14.5   13.5    65.2     52.1%    19.1%    -114   (BetOnline.ag) 22.4     6.39     LOW   
Naji Marshall             DAL    Under  13.5   11.8

In [9]:
# =============================================================================
# 6. SAVE RESULTS
# =============================================================================

# Prepare output DataFrame
output_df = singles_df[[
    'player', 'team', 'line', 'side', 'projection', 'std',
    'model_prob', 'fair_prob', 'pp_odds', 'pp_implied',
    'best_us_odds', 'best_us_book',
    'edge_vs_fair', 'edge_vs_pp', 'ev', 'ev_percent', 'kelly_quarter',
    'usage_adj', 'usage_reason', 'L-5', 'L-10', 'L-15'
]].rename(columns={
    'player': 'NAME',
    'team': 'TEAM',
    'line': 'LINE',
    'side': 'SIDE',
    'projection': 'PREDICTION',
    'std': 'STD',
    'model_prob': 'MODEL_PROB',
    'fair_prob': 'FAIR_PROB',
    'pp_odds': 'PRIZEPICKS_ODDS',
    'pp_implied': 'PRIZEPICKS_IMPLIED',
    'best_us_odds': 'BEST_US_ODDS',
    'best_us_book': 'BEST_US_BOOK',
    'edge_vs_fair': 'EDGE_VS_FAIR',
    'edge_vs_pp': 'EDGE_VS_PP',
    'ev': 'EV',
    'ev_percent': 'EV_PERCENT',
    'kelly_quarter': 'KELLY_QUARTER',
    'usage_adj': 'USAGE_ADJ',
    'usage_reason': 'USAGE_REASON',
    'L-5': 'L5',
    'L-10': 'L10',
    'L-15': 'L15'
})

# Save
from datetime import datetime
today = datetime.now().strftime('%Y-%m-%d')
output_path = f'data/props/ev_analysis/underdog.csv'
output_df.to_csv(output_path, index=False)
print(f"\n✓ Saved to {output_path}")
print(f"✓ Total picks: {len(output_df)}")
print(f"✓ Picks with >2% edge: {len(singles_df[singles_df['edge_vs_fair'] > 0.02])}")
print(f"✓ Picks with >5% edge: {len(singles_df[singles_df['edge_vs_fair'] > 0.05])}")
print(f"✓ Picks using default -137: {len(singles_df[singles_df['best_us_book'] == 'Default (-137)'])}")
print(f"✓ Picks with usage adjustments: {len(singles_df[singles_df['usage_adj'] != 1.0])}")


✓ Saved to data/props/ev_analysis/underdog.csv
✓ Total picks: 41
✓ Picks with >2% edge: 23
✓ Picks with >5% edge: 18
✓ Picks using default -137: 0
✓ Picks with usage adjustments: 36


In [10]:
singles_df[['player', 'side', 'line', 'projection', 'model_prob','ev_percent', 'kelly_quarter', 'L-5', 'L-10', 'L-15']].sort_values('ev_percent', ascending=False).head(20)

,player,side,line,projection,model_prob,ev_percent,kelly_quarter,L-5,L-10,L-15
30,Ja Morant,Over,19.5,24.9,0.822,60.53,0.1589,40.0,40.0,38.5
24,Kyle Filipowski,Under,12.5,10.9,0.783,54.24,0.1397,60.0,70.0,73.3
10,Tyler Herro,Over,21.5,25.5,0.805,52.39,0.1467,60.0,66.7,66.7
17,Anthony Davis,Under,24.5,19.9,0.802,50.00,0.1438,60.0,50.0,54.5
29,Aaron Holiday,Over,7.5,10.8,0.610,28.10,0.0639,80.0,80.0,66.7
21,Max Christie,Over,10.5,12.2,0.615,24.31,0.0596,40.0,50.0,53.3
28,Cameron Johnson,Under,14.5,13.5,0.652,22.43,0.0639,60.0,40.0,53.3
20,Ace Bailey,Under,12.5,11.3,0.625,19.32,0.0531,40.0,40.0,40.0
39,Mikal Bridges,Over,14.5,15.8,0.625,18.30,0.0512,60.0,60.0,66.7
0,Anfernee Simons,Over,11.5,12.7,0.600,17.69,0.0460,60.0,60.0,53.3
